In [1]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import (
    load_config,
    extract_mean_amplitudes,
    run_paired_tests,
    run_anova,
    plot_results,
)

cfg = load_config('../../../configs/eye_eeg_simul.yaml')
cfg_erp = load_config('../../../configs/erp_position.yaml')

print("Setup OK")

Setup OK


In [2]:
import pandas as pd

# ROIs and time windows
rois = {
    'posterior_left':  ['P3', 'P7', 'O1'],
    'posterior_right': ['P4', 'P8', 'O2'],
}

time_windows = {
    'early': [0.20, 0.30],
    'late':  [0.30, 0.50],
}

# Extract all 16 granular conditions
df_raw = extract_mean_amplitudes(
    cfg, window_name='cue',
    rois=rois,
    time_windows=time_windows,
    conditions=None,  # all non-contrast conditions
    save_csv=False,
    verbose=True,
)

# Collapse pos1+pos2 -> left, pos3+pos4 -> right
groupings = {
    'spatial_target_left':       ['spatial_target_pos1', 'spatial_target_pos2'],
    'spatial_target_right':      ['spatial_target_pos3', 'spatial_target_pos4'],
    'spatial_distractor_left':   ['spatial_distractor_dig1', 'spatial_distractor_dig2'],
    'spatial_distractor_right':  ['spatial_distractor_dig3', 'spatial_distractor_dig4'],
    'symbolic_target_left':      ['symbolic_target_dig1', 'symbolic_target_dig2'],
    'symbolic_target_right':     ['symbolic_target_dig3', 'symbolic_target_dig4'],
    'symbolic_distractor_left':  ['symbolic_distractor_pos1', 'symbolic_distractor_pos2'],
    'symbolic_distractor_right': ['symbolic_distractor_pos3', 'symbolic_distractor_pos4'],
}

rows = []
for grp_name, components in groupings.items():
    sub = df_raw[df_raw['condition'].isin(components)]
    collapsed = sub.groupby(['subject', 'window', 'roi', 'channels',
                             'time_window', 'tmin', 'tmax'], as_index=False).agg(
        mean_amp_uv=('mean_amp_uv', 'mean')
    )
    collapsed['condition'] = grp_name
    rows.append(collapsed)

df = pd.concat(rows, ignore_index=True)
print(f"\nCollapsed: {df['condition'].nunique()} conditions, "
      f"{df['subject'].nunique()} subjects, {len(df)} rows")
print(df.groupby('condition')['subject'].nunique().to_string())

[find_subjects] excluded 8 subject(s): ['subj13', 'subj15', 'subj16', 'subj17', 'subj2', 'subj20', 'subj28', 'subj7']

Extracted 1920 rows from 30 subjects.
   ROIs: ['posterior_left', 'posterior_right']
   Time windows: ['early', 'late']
   Conditions: ['spatial_target_pos1', 'spatial_target_pos2', 'spatial_target_pos3', 'spatial_target_pos4', 'spatial_distractor_dig1', 'spatial_distractor_dig2', 'spatial_distractor_dig3', 'spatial_distractor_dig4', 'symbolic_target_dig1', 'symbolic_target_dig2', 'symbolic_target_dig3', 'symbolic_target_dig4', 'symbolic_distractor_pos1', 'symbolic_distractor_pos2', 'symbolic_distractor_pos3', 'symbolic_distractor_pos4']

Collapsed: 8 conditions, 30 subjects, 960 rows
condition
spatial_distractor_left      30
spatial_distractor_right     30
spatial_target_left          30
spatial_target_right         30
symbolic_distractor_left     30
symbolic_distractor_right    30
symbolic_target_left         30
symbolic_target_right        30


In [4]:
from statsmodels.stats.anova import AnovaRM

# Parse condition names into factors
df['cue_type'] = df['condition'].apply(lambda x: 'spatial' if x.startswith('spatial') else 'symbolic')
df['role'] = df['condition'].apply(lambda x: 'target' if 'target' in x else 'distractor')
df['side'] = df['condition'].apply(lambda x: 'left' if x.endswith('left') else 'right')

# Verify the parsing
print("Factor levels:")
print(f"   cue_type: {df['cue_type'].unique().tolist()}")
print(f"   role:     {df['role'].unique().tolist()}")
print(f"   side:     {df['side'].unique().tolist()}")
print(f"   N per cell: {len(df) // df[['cue_type','role','side','roi','time_window']].drop_duplicates().shape[0]}")

# Run 2x2x2 RM-ANOVA per ROI per time window
for (roi, tw), group in df.groupby(['roi', 'time_window']):
    print(f"\n{'='*60}")
    print(f"ROI: {roi} | Time window: {tw}")
    print(f"{'='*60}")
    
    aov = AnovaRM(
        group, depvar='mean_amp_uv', subject='subject',
        within=['cue_type', 'role', 'side']
    ).fit()
    print(aov.anova_table.round(4))

Factor levels:
   cue_type: ['spatial', 'symbolic']
   role:     ['target', 'distractor']
   side:     ['left', 'right']
   N per cell: 30

ROI: posterior_left | Time window: early

ROI: posterior_left | Time window: late

ROI: posterior_right | Time window: early

ROI: posterior_right | Time window: late


In [5]:
from statsmodels.stats.multitest import multipletests
import pandas as pd

# Collect all ANOVA results for FDR correction
anova_rows = []
for (roi, tw), group in df.groupby(['roi', 'time_window']):
    aov = AnovaRM(
        group, depvar='mean_amp_uv', subject='subject',
        within=['cue_type', 'role', 'side']
    ).fit()
    
    for effect, row in aov.anova_table.iterrows():
        anova_rows.append({
            'roi': roi,
            'time_window': tw,
            'effect': effect,
            'F': row['F Value'],
            'df1': row['Num DF'],
            'df2': row['Den DF'],
            'p': row['Pr > F'],
        })

anova_df = pd.DataFrame(anova_rows)

# FDR across ALL tests (7 effects × 4 ROI×TW = 28 tests)
_, anova_df['p_fdr'], _, _ = multipletests(anova_df['p'], method='fdr_bh')
anova_df['sig_fdr'] = anova_df['p_fdr'] < 0.05

print(f"FDR correction across {len(anova_df)} tests\n")
print(anova_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Show only significant effects
sig = anova_df[anova_df['sig_fdr']]
print(f"\n--- Significant after FDR ({len(sig)}) ---")
if len(sig) > 0:
    print(sig.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
else:
    print("None survived FDR correction.")

FDR correction across 28 tests

            roi time_window             effect       F    df1     df2      p  p_fdr  sig_fdr
 posterior_left       early           cue_type  1.4171 1.0000 29.0000 0.2435 0.4871    False
 posterior_left       early               role  2.4381 1.0000 29.0000 0.1293 0.4022    False
 posterior_left       early               side  1.6314 1.0000 29.0000 0.2116 0.4558    False
 posterior_left       early      cue_type:role  2.2357 1.0000 29.0000 0.1457 0.4079    False
 posterior_left       early      cue_type:side  0.0613 1.0000 29.0000 0.8062 0.9485    False
 posterior_left       early          role:side  3.2748 1.0000 29.0000 0.0807 0.3767    False
 posterior_left       early cue_type:role:side  9.7682 1.0000 29.0000 0.0040 0.0562    False
 posterior_left        late           cue_type  3.3948 1.0000 29.0000 0.0756 0.3767    False
 posterior_left        late               role  0.4133 1.0000 29.0000 0.5254 0.7355    False
 posterior_left        late           

In [6]:
contrasts = [
    ('spatial_target_left',      'spatial_target_right'),
    ('spatial_distractor_left',  'spatial_distractor_right'),
    ('symbolic_target_left',     'symbolic_target_right'),
    ('symbolic_distractor_left', 'symbolic_distractor_right'),
]

results = run_paired_tests(
    df,
    contrasts=contrasts,
    cfg=cfg, window_name='cue',
    save_csv=True, verbose=True,
)


Paired tests: 4 contrast(s), FDR across 16 tests total.

                                             contrast             roi time_window   tmin   tmax  n  mean_diff_uv  sd_diff_uv  cohens_d       t    p_t  p_t_fdr  sig_t_fdr        w    p_w  p_w_fdr  sig_w_fdr
          spatial_target_left_vs_spatial_target_right  posterior_left       early 0.2000 0.3000 30        0.6246      1.0967    0.5695  3.1195 0.0041   0.0652      False 105.0000 0.0076   0.1218      False
          spatial_target_left_vs_spatial_target_right  posterior_left        late 0.3000 0.5000 30       -0.1498      1.1128   -0.1346 -0.7372 0.4669   0.6225      False 192.0000 0.4161   0.5548      False
          spatial_target_left_vs_spatial_target_right posterior_right       early 0.2000 0.3000 30       -0.4386      1.1608   -0.3778 -2.0694 0.0475   0.1889      False 135.0000 0.0449   0.1796      False
          spatial_target_left_vs_spatial_target_right posterior_right        late 0.3000 0.5000 30        0.3298      

In [7]:
# Factors are already parsed from earlier
# df already has: subject, roi, channels, condition, time_window, tmin, tmax, mean_amp_uv, cue_type, role, side

# Save long format (ready for R, JASP, jamovi, SPSS)
from eeg_toolkit.evoked import _get_erp_dir

out_path = _get_erp_dir(cfg) / "stats" / "factorial_amplitudes_for_export.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

export = df[['subject', 'roi', 'time_window', 'tmin', 'tmax',
             'cue_type', 'role', 'side', 'condition', 'mean_amp_uv']].copy()
export = export.sort_values(['subject', 'roi', 'time_window', 'cue_type', 'role', 'side'])
export.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Shape: {export.shape}")
print(f"\nColumns: {export.columns.tolist()}")
print(f"\nHead:")
export.head(8)

Saved: C:\Users\juapa\OneDrive\Documentos\Proyecto doctorado\Experimentos_proyecto_tesis\Experimentos Eye Tracker\analisis\analisis_eeg\group_results\erp\stats\factorial_amplitudes_for_export.csv
Shape: (960, 10)

Columns: ['subject', 'roi', 'time_window', 'tmin', 'tmax', 'cue_type', 'role', 'side', 'condition', 'mean_amp_uv']

Head:


,subject,roi,time_window,tmin,tmax,cue_type,role,side,condition,mean_amp_uv
240,subj10,posterior_left,early,0.2,0.3,spatial,distractor,left,spatial_distractor_left,-0.310950
360,subj10,posterior_left,early,0.2,0.3,spatial,distractor,right,spatial_distractor_right,-0.224149
0,subj10,posterior_left,early,0.2,0.3,spatial,target,left,spatial_target_left,-0.845529
120,subj10,posterior_left,early,0.2,0.3,spatial,target,right,spatial_target_right,0.181060
720,subj10,posterior_left,early,0.2,0.3,symbolic,distractor,left,symbolic_distractor_left,0.434061
840,subj10,posterior_left,early,0.2,0.3,symbolic,distractor,right,symbolic_distractor_right,0.966411
480,subj10,posterior_left,early,0.2,0.3,symbolic,target,left,symbolic_target_left,0.839902
600,subj10,posterior_left,early,0.2,0.3,symbolic,target,right,symbolic_target_right,0.643948
